# 00 · Configuración del proyecto

**Fase del plan:** F0 – Decisiones de diseño (preparación del entorno).

**Objetivo:** centralizar en un solo lugar las rutas a los datos, los parámetros de diseño del análisis de la Pregunta 6 y las funciones auxiliares (`guardar_fig()`, `guardar_tabla()`), para que todos los notebooks posteriores las carguen sin reescribir código.

**Entradas:** archivos originales en `data/` (no se modifican).

**Salidas:** `output/data/config.rds`, una lista con rutas, parámetros y helpers. Cada notebook la carga con:

```r
cfg <- readRDS(here::here("output/data/config.rds")); list2env(cfg, envir = globalenv()); set.seed(SEED)
```


## Rutas a los datos

Las rutas se construyen con `here::here()`, que resuelve la raíz del proyecto (marcada con el archivo `.here`) sin importar desde qué carpeta se ejecute el notebook. Se verifica que los cuatro archivos existen.


In [1]:
library(here)

RUTA_MEDIUM    <- here::here("data", "WPP2024_Demographic_Indicators_Medium_csv.gz")
RUTA_VARIANTES <- here::here("data", "WPP2024_Demographic_Indicators_OtherVariants_csv.gz")
RUTA_NOTAS     <- here::here("data", "WPP2024_Demographic_Indicators_notes.csv")
RUTA_SUPLEM    <- here::here("data", "WPP2024_Supplement_Births_and_LE60.csv")

rutas <- list(RUTA_MEDIUM = RUTA_MEDIUM, RUTA_VARIANTES = RUTA_VARIANTES,
              RUTA_NOTAS = RUTA_NOTAS, RUTA_SUPLEM = RUTA_SUPLEM)
stopifnot(all(file.exists(unlist(rutas))))
print(sapply(rutas, function(r) sub(paste0(here::here(), "/"), "", r)))


here() starts at /Users/santiagosantosojeda/Downloads/2026 ASA DataQuest



                                               RUTA_MEDIUM 
       "data/WPP2024_Demographic_Indicators_Medium_csv.gz" 
                                            RUTA_VARIANTES 
"data/WPP2024_Demographic_Indicators_OtherVariants_csv.gz" 
                                                RUTA_NOTAS 
           "data/WPP2024_Demographic_Indicators_notes.csv" 
                                               RUTA_SUPLEM 
             "data/WPP2024_Supplement_Births_and_LE60.csv" 


## Parámetros de diseño (F0)

Las decisiones de la sección 2 del plan quedan como parámetros, para poder cambiarlas en la sensibilidad (F8) sin reescribir código.

| Decisión | Valor principal | Sensibilidad (F8) |
|---|---|---|
| Horizonte | 1950–2023 (estimaciones) | 1950–2100 (incluye proyección) |
| Indicadores núcleo | `TFR`, `LEx`, `CNMR` | Ampliado: + `MAC`, `MedianAgePop` |
| Países pequeños | Excluir si `TPopulation1July` 2023 < 100 (miles) → quedan 200 países | Incluir los 237 |
| Ponderación | Ninguna (unidad = país) | — |
| Transformación de CNMR | `asinh(CNMR)` | CNMR winsorizado al p1–p99 |
| Normalización | Chiou et al. (2014): $Z = (X-\hat\mu)/\sqrt{\hat v}$ | Sin normalizar, solo centrado |
| Selección de $L$ | FVE ≥ 90 % | FVE ≥ 95 % |
| Clustering principal | GMM (`mclust`) sobre los scores, $G$ por BIC | Ward, k-means, `funHDDC` |

**Justificación de los indicadores**, con Spearman sobre países 1950–2023:
- TFR, CBR y NRR tienen $\rho \ge 0.95$ entre sí, así que se usa solo TFR.
- LEx, IMR, Q5 y LE65 tienen $|\rho| \ge 0.96$ entre sí, así que se usa solo LEx.
- CNMR es casi independiente del resto ($|\rho| \le 0.23$) y forma una dimensión propia.
- MAC es poco redundante ($|\rho| \le 0.32$).
- MedianAgePop es el resultado de los tres motores; solo entra en la versión ampliada.

Se descartan `PopGrowthRate` y `NatChangeRT` por identidades contables: `NatChangeRT` = `CBR − CDR` exacto, y `PopGrowthRate` ≈ (`NatChangeRT` + `CNMR`)/10 con $r = 0.9993$.

**Correspondencia con los parámetros:**
- `HORIZONTE` / `HORIZONTE_SENS`: horizonte principal (solo estimaciones, para no agrupar según los supuestos de proyección de la ONU) y de sensibilidad.
- `IND_NUCLEO` / `IND_AMPLIADO`: un indicador por dimensión demográfica (fecundidad, mortalidad, migración); el ampliado añade `MAC` y `MedianAgePop`.
- `MIN_POP_MILES`, `ANIO_REF_POP`: umbral de 100 mil habitantes en 2023; por debajo, las tasas (sobre todo CNMR) son muy ruidosas.
- `TRANSF_CNMR`: `asinh` amortigua los picos de CNMR y conserva el signo; alternativas `"winsor"` (p1–p99) y `"ninguna"`.
- `FVE_UMBRAL`: número de componentes del MFPCA, $L=\min\{L:\mathrm{FVE}(L)\ge 0.90\}$.
- `G_RANGO`: número de grupos evaluados por BIC, $G\in\{2,\dots,8\}$.
- `B_BOOT`: réplicas bootstrap para la estabilidad (`fpc::clusterboot`, Jaccard medio > 0.75 = estable).
- `SEED`: semilla para reproducibilidad.


In [2]:
HORIZONTE      <- c(1950, 2023)
HORIZONTE_SENS <- c(1950, 2100)
IND_NUCLEO     <- c("TFR", "LEx", "CNMR")
IND_AMPLIADO   <- c("TFR", "LEx", "CNMR", "MAC", "MedianAgePop")
MIN_POP_MILES  <- 100
ANIO_REF_POP   <- 2023
TRANSF_CNMR    <- "asinh"   # alternativas: "winsor", "ninguna"
FVE_UMBRAL     <- 0.90
G_RANGO        <- 2:8
B_BOOT         <- 500
SEED           <- 20261014

parametros <- list(HORIZONTE = HORIZONTE, HORIZONTE_SENS = HORIZONTE_SENS,
                   IND_NUCLEO = IND_NUCLEO, IND_AMPLIADO = IND_AMPLIADO,
                   MIN_POP_MILES = MIN_POP_MILES, ANIO_REF_POP = ANIO_REF_POP,
                   TRANSF_CNMR = TRANSF_CNMR, FVE_UMBRAL = FVE_UMBRAL,
                   G_RANGO = G_RANGO, B_BOOT = B_BOOT, SEED = SEED)

stopifnot(TRANSF_CNMR %in% c("asinh", "winsor", "ninguna"),
          all(IND_NUCLEO %in% IND_AMPLIADO),
          HORIZONTE_SENS[2] <= 2100,
          FVE_UMBRAL > 0, FVE_UMBRAL < 1)
str(parametros)


List of 11
 $ HORIZONTE     : num [1:2] 1950 2023
 $ HORIZONTE_SENS: num [1:2] 1950 2100
 $ IND_NUCLEO    : chr [1:3] "TFR" "LEx" "CNMR"
 $ IND_AMPLIADO  : chr [1:5] "TFR" "LEx" "CNMR" "MAC" ...
 $ MIN_POP_MILES : num 100
 $ ANIO_REF_POP  : num 2023
 $ TRANSF_CNMR   : chr "asinh"
 $ FVE_UMBRAL    : num 0.9
 $ G_RANGO       : int [1:7] 2 3 4 5 6 7 8
 $ B_BOOT        : num 500
 $ SEED          : num 20261014


## Funciones auxiliares

- `guardar_fig(p, nombre, w = 9, h = 6)`: guarda un gráfico `ggplot2` en `output/figures/` como PDF y PNG (300 dpi), con ancho `w` y alto `h` en pulgadas. Devuelve el gráfico de forma invisible para que también se pueda mostrar inline.
- `guardar_tabla(df, nombre, tex = FALSE)`: guarda un data frame como CSV en `output/tables/` y, si `tex = TRUE`, también como tabla LaTeX (`knitr::kable(format = "latex", booktabs = TRUE)`).

Las funciones usan llamadas con `paquete::` para no depender de los paquetes cargados en el notebook que las use. Se prueban con una figura y una tabla de prueba, que luego se borran.


In [3]:
guardar_fig <- function(p, nombre, w = 9, h = 6) {
  dir_fig <- here::here("output", "figures")
  dir.create(dir_fig, recursive = TRUE, showWarnings = FALSE)
  rutas <- file.path(dir_fig, paste0(nombre, c(".pdf", ".png")))
  ggplot2::ggsave(rutas[1], plot = p, width = w, height = h, device = "pdf", encoding = "WinAnsi.enc")
  ggplot2::ggsave(rutas[2], plot = p, width = w, height = h, dpi = 300, bg = "white")
  invisible(p)
}

guardar_tabla <- function(df, nombre, tex = FALSE) {
  dir_tab <- here::here("output", "tables")
  dir.create(dir_tab, recursive = TRUE, showWarnings = FALSE)
  readr::write_csv(df, file.path(dir_tab, paste0(nombre, ".csv")))
  if (tex) {
    latex <- knitr::kable(df, format = "latex", booktabs = TRUE)
    writeLines(latex, file.path(dir_tab, paste0(nombre, ".tex")))
  }
  invisible(df)
}

# Prueba de los helpers (los archivos de prueba se eliminan)
library(ggplot2)
p_prueba <- ggplot(data.frame(x = 1:10, y = (1:10)^2), aes(x, y)) +
  geom_line() + theme_minimal(base_size = 11) +
  labs(title = "Prueba de guardado: año, población, ñandú", caption = "Fuente: UN WPP 2024")
guardar_fig(p_prueba, "zz_prueba")
guardar_tabla(data.frame(a = 1:3, b = c("x", "y", "ñ")), "zz_prueba", tex = TRUE)

archivos_prueba <- c(here::here("output/figures", c("zz_prueba.pdf", "zz_prueba.png")),
                     here::here("output/tables",  c("zz_prueba.csv", "zz_prueba.tex")))
stopifnot(all(file.exists(archivos_prueba)), all(file.size(archivos_prueba) > 0))
print(data.frame(archivo = basename(archivos_prueba), bytes = file.size(archivos_prueba)))
invisible(file.remove(archivos_prueba))


        archivo bytes
1 zz_prueba.pdf  4444
2 zz_prueba.png 88560
3 zz_prueba.csv    17
4 zz_prueba.tex    98


## Guardado de la configuración

Se reúnen rutas, parámetros y helpers en una sola lista y se guarda en `output/data/config.rds`. Luego se vuelve a leer en un entorno limpio para verificar que el archivo es válido y que los helpers funcionan tras cargarlos.


In [4]:
RUTA_CONFIG <- here::here("output", "data", "config.rds")
config <- c(rutas, parametros,
            list(guardar_fig = guardar_fig, guardar_tabla = guardar_tabla))
saveRDS(config, RUTA_CONFIG)

# Verificación: relectura en un entorno aislado
cfg_leida <- readRDS(RUTA_CONFIG)
stopifnot(identical(names(cfg_leida), names(config)),
          identical(cfg_leida[names(parametros)], parametros),
          identical(cfg_leida[names(rutas)], rutas),
          is.function(cfg_leida$guardar_fig), is.function(cfg_leida$guardar_tabla))
env_prueba <- new.env()
invisible(list2env(cfg_leida, envir = env_prueba))
stopifnot(all(file.exists(unlist(mget(names(rutas), envir = env_prueba)))))

cat("config.rds guardado en:", sub(paste0(here::here(), "/"), "", RUTA_CONFIG), "\n")
cat("Elementos:", paste(names(cfg_leida), collapse = ", "), "\n")


config.rds guardado en: output/data/config.rds 


Elementos: RUTA_MEDIUM, RUTA_VARIANTES, RUTA_NOTAS, RUTA_SUPLEM, HORIZONTE, HORIZONTE_SENS, IND_NUCLEO, IND_AMPLIADO, MIN_POP_MILES, ANIO_REF_POP, TRANSF_CNMR, FVE_UMBRAL, G_RANGO, B_BOOT, SEED, guardar_fig, guardar_tabla 


## Conclusiones

- Las rutas a los cuatro archivos de `data/` existen y se resuelven con `here::here()` desde cualquier carpeta del proyecto (raíz marcada con `.here`).
- Los parámetros de diseño de F0 quedan centralizados: horizonte 1950–2023, indicadores núcleo `TFR`, `LEx`, `CNMR`, umbral de 100 mil habitantes en 2023, `asinh` para CNMR, FVE ≥ 90 %, $G \in \{2,\dots,8\}$, $B = 500$ y semilla `20261014`.
- `guardar_fig()` y `guardar_tabla()` se probaron con éxito (PDF + PNG 300 dpi; CSV + LaTeX). El PDF usa el dispositivo `pdf` estándar porque `cairo_pdf` no carga en este kernel; se usa la codificación `WinAnsi.enc` para que las tildes, la ñ y la raya (–) se vean bien.
- Todo queda en `output/data/config.rds`, que se relee y valida al final. Los notebooks siguientes empiezan con `cfg <- readRDS(here::here("output/data/config.rds")); list2env(cfg, envir = globalenv()); set.seed(SEED)`.
